# M6 Treatment Runs: Arabic Flood-Crisis Tweet Classification

This notebook runs the three-agent treatment (Relevance Agent, Information-Type Agent, Judge) on the frozen golden set
with the same three seeds as the baseline, then scores it with the same code.

Run sections 1 to 4 once per session. Sections 5 and 6 are one-off checks. Section 7 is the main run: one seed at a time.
Results are written to Google Drive, so a run that stops (quota, closed tab) can continue in a later session.

## 1. Get the code

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/arabic-flood-m4-baseline"
    if not os.path.exists(REPO):
        !git clone -q https://github.com/Maya-mohammed/arabic-flood-m4-baseline.git {REPO}
    %cd {REPO}
    !git pull -q
    !pip install -q -r requirements.txt pytest
!git log -1 --format="Code version: %h %ad %s" --date=short

## 2. Offline unit tests

In [ ]:
!python -m pytest -q tests

## 3. Gemini key and results folder

The key is read from Colab Secrets (`GEMINI_API_KEY`) and never printed.
Results go to `flood_m6_runs` in your Google Drive. The files hold IDs, labels and usage counts only, no tweet text.

In [ ]:
from google.colab import userdata, drive

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Gemini key loaded:", bool(os.environ["GEMINI_API_KEY"]))

drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/flood_m6_runs"
os.makedirs(RUNS, exist_ok=True)
print("Results folder:", RUNS)

## 4. Data and golden set

Downloads the corpus from the authors' repository, rebuilds the splits, and rebuilds the golden set.
`make_golden_set.py` stops if its sample differs from the frozen `m5/golden_set_ids.csv`.

In [ ]:
!test -d source_data || git clone -q --depth 1 https://github.com/alaa-a-a/Arabic-Twitter-corpus-for-flood-detection.git source_data
!mkdir -p private_data && cp source_data/ArabicFloods.csv private_data/
!python prepare_data.py --csv private_data/ArabicFloods.csv
!python make_golden_set.py

## 5. Check: same results on this account (15 calls)

Runs the **baseline** on the first 5 golden-set cases with seed 42 and compares them with the saved M5 run.
If all 5 match at every stage, the account and the date do not change the model's output, and the treatment runs
can be compared with the saved baseline runs.

In [ ]:
import pandas as pd

check = f"{RUNS}/check_baseline_seed_42.csv"
!python evaluate.py --system baseline --cases private_data/golden_set.csv --out {check} --limit 5 --seed 42 --pause 10 --resume

stages = ["initial_relevance", "initial_information_type", "review_relevance",
          "review_information_type", "pred_relevance", "pred_information_type"]
new = pd.read_csv(check, dtype=str).set_index("case_id")
saved = pd.read_csv("m5/runs/baseline/seed_42.csv", dtype=str).set_index("case_id")
same = (new[stages] == saved.loc[new.index, stages]).all(axis=1)
print(f"Cases identical to the saved M5 seed 42 run at every stage: {int(same.sum())} of {len(new)}")

## 6. Check: treatment pipeline on development tweets (24 calls)

Runs the treatment on the 8 development cases used in M4 (one per label pair).
This checks format and pipeline only. The prompts are not changed because of these labels.

In [ ]:
dev_check = f"{RUNS}/check_treatment_dev.csv"
!python evaluate.py --system treatment --cases private_data/normal_cases.csv --out {dev_check} --seed 42 --pause 10 --resume
pd.read_csv(dev_check)[["case_id", "initial_relevance", "initial_information_type",
                        "pred_relevance", "pred_information_type", "repairs", "error"]]

## 7. Main run: treatment on the golden set

Set `SEED` and run the cell. One seed is 140 tweets x 3 calls = 420 planned calls.
If the daily quota runs out, the run stops and keeps what it has. Run the same cell again after the quota resets
(11:00 Riyadh time) and it continues from the next tweet. Do the three seeds: 42, 123, 2026.

In [ ]:
SEED = 42   # then 123, then 2026

out = f"{RUNS}/seed_{SEED}.csv"
!python evaluate.py --system treatment --cases private_data/golden_set.csv --out {out} --seed {SEED} --pause 10 --resume
done = len(pd.read_csv(out)) if os.path.exists(out) else 0
print(f"Seed {SEED}: {done} of 140 cases saved in {out}")

## 8. Score the treatment (after all three seeds are complete)

In [ ]:
!mkdir -p m6/runs/treatment
!cp {RUNS}/seed_42.csv {RUNS}/seed_123.csv {RUNS}/seed_2026.csv m6/runs/treatment/
!python score_runs.py --condition treatment \
    --runs m6/runs/treatment/seed_42.csv m6/runs/treatment/seed_123.csv m6/runs/treatment/seed_2026.csv \
    --out m6/results/treatment --price-input 0.30 --price-output 2.50